# rag-starter: Your First RAG Pipeline (Colab Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salahuddinuqaili/rag-starter/blob/main/notebooks/rag_starter_colab.ipynb)

This notebook walks you through building a RAG (Retrieval-Augmented Generation) pipeline from scratch.
Since Ollama can't run in Colab, we use **Groq's free API** for the LLM and embeddings.

No local setup required — just click "Run all" and follow along.

## Step 1: Install dependencies

We need ChromaDB for vector storage and the Groq/OpenAI client for LLM calls.

In [ ]:
!pip install -q chromadb openai groq

## Step 2: Set your Groq API key

Get a free API key from [console.groq.com](https://console.groq.com). No credit card required.
Paste it below — it stays in this notebook session only.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

## Step 3: Define sample documents

These are the same sample documents from the rag-starter repo, inlined here so the
notebook is self-contained.

In [ ]:
documents = {
    "remote-work-best-practices.md": """
Remote work requires intentional communication. Teams should default to asynchronous
methods like written updates and shared documents rather than scheduling meetings for
every discussion. When meetings are necessary, always circulate an agenda beforehand
and share notes within 24 hours. Keep your calendar visible so teammates in other
timezones know when you're available. Documentation culture means writing decisions
down — not in Slack threads that disappear, but in shared docs that persist.
""",
    "intro-to-machine-learning.md": """
Machine learning is the practice of teaching computers to find patterns in data.
Supervised learning uses labelled examples — like emails tagged as spam or not spam —
to train a model that can classify new data. Unsupervised learning discovers structure
without labels, such as grouping customers by purchasing behaviour. The quality of your
training data matters more than the sophistication of your algorithm. Overfitting is when
a model memorises its training data instead of learning general patterns — like a student
who memorises exam answers but can't solve new problems.
""",
    "berlin-travel-guide.md": """
Berlin is a city of contrasts. Kreuzberg pulses with street art and independent cafes.
Mitte houses world-class museums along the Spree. Prenzlauer Berg is where young families
brunch on weekends. Neukolln has the best nightlife if you know where to look. Get around
on the U-Bahn and S-Bahn — a day pass is cheap and covers the whole city. Don't skip the
currywurst, try a proper doner kebab, and find a third-wave coffee shop. Cash is still
king in many smaller places, and almost everything is closed on Sundays.
""",
    "what-is-rag.md": """
RAG (Retrieval-Augmented Generation) solves a fundamental problem: language models don't
know about your private documents. Instead of retraining the entire model, RAG retrieves
relevant context from your documents and includes it in the prompt. The pipeline works in
five steps: load documents, split them into chunks, convert chunks into embeddings, store
embeddings in a vector database, then retrieve the most relevant chunks when answering a
question. RAG is cheaper and faster than fine-tuning, and your data stays private.
"""
}

print(f"Loaded {len(documents)} sample documents")

## Step 4: Chunk the documents

We split each document into smaller overlapping pieces. This helps retrieval find
the most relevant section instead of returning an entire document.

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks."""
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []
all_metadatas = []

for filename, content in documents.items():
    chunks = chunk_text(content)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({"source": filename, "chunk_index": i})

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")

## Step 5: Generate embeddings

Embeddings are lists of numbers that capture the meaning of text. We use Groq's API
to generate them. Similar text gets similar numbers — that's how retrieval works.

In [ ]:
from openai import OpenAI

# Groq is compatible with the OpenAI client
embed_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Convert texts to embedding vectors using Groq."""
    response = embed_client.embeddings.create(
        model="nomic-embed-text-v1_5",
        input=texts
    )
    return [item.embedding for item in response.data]

embeddings = embed_texts(all_chunks)
print(f"Generated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

## Step 6: Store in ChromaDB

ChromaDB is an embedded vector database — it stores your embeddings so you can
search them by meaning, not just keywords.

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("rag_demo", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=all_chunks,
    embeddings=embeddings,
    metadatas=all_metadatas,
)

print(f"Stored {collection.count()} chunks in ChromaDB")

## Step 7: Query with RAG

Now the fun part! We embed the question, find the most similar chunks, and ask
the LLM to answer based only on those chunks.

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask(question: str, top_k: int = 3) -> str:
    """Full RAG pipeline: embed question, retrieve chunks, generate answer."""
    # Embed the question
    query_embedding = embed_texts([question])[0]

    # Retrieve similar chunks
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)

    # Build context from retrieved chunks
    context = "\n---\n".join(results["documents"][0])

    # Generate answer
    prompt = f"""You are a helpful assistant. Answer the question based ONLY on the following context.
If the context doesn't contain enough information, say so.

Context:
{context}

Question: {question}

Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.choices[0].message.content
    sources = set(m["source"] for m in results["metadatas"][0])

    print(f"Question: {question}\n")
    print(f"Answer: {answer}\n")
    print(f"Sources: {', '.join(sources)}\n")
    print("-" * 60)
    return answer

# Ask three cross-topic questions
ask("What are the best practices for async communication in remote teams?")
ask("How does supervised learning differ from unsupervised learning?")
ask("What neighbourhoods should I visit in Berlin?")

## Try your own!

Type any question below. The RAG pipeline will search the sample documents and
generate an answer grounded in the retrieved context.

In [ ]:
your_question = input("Your question: ")
if your_question.strip():
    ask(your_question)